In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
from unittest.mock import AsyncMock, Mock, patch
import httpx
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from dotenv import load_dotenv
from matplotlib.colors import ListedColormap
from shapely.geometry import shape, Polygon
import gc

# Make sure your dotenv file has the following defined:
load_dotenv()
GIT_FOLDER = os.environ['GIT_FOLDER']
TITILER_URL = os.environ['TITILER_URL']
TITILER_API_KEY = os.environ['TITILER_API_KEY']
API_KEY = os.environ['API_KEY']
MODEL_PATH_LOCAL = os.environ['MODEL_PATH_LOCAL']

if not (GIT_FOLDER and TITILER_URL and TITILER_API_KEY and MODEL_PATH_LOCAL):
    print("ERRROR: Failed to find all the necessary environment variables!!!")
    # Note, you must restart the kernel if you want to load new environment variables

if GIT_FOLDER not in sys.path:
    sys.path.append(GIT_FOLDER)
print(sys.path)

from ensembler import Task, Ensemblers

In [ ]:
from cerulean_cloud.models import *
from cerulean_cloud.tiling import TMS, offset_bounds_from_base_tiles
from cerulean_cloud.titiler_client import TitilerClient
from cerulean_cloud.cloud_run_orchestrator.clients import img_array_to_b64_image
from cerulean_cloud.cloud_run_orchestrator.schema import OrchestratorInput
from cerulean_cloud.cloud_run_orchestrator.handler import *#_orchestrate, get_tiler, get_titiler_client, get_roda_sentinelhub_client, get_database_engine
from cerulean_cloud.cloud_run_offset_tiles.schema import * #import InferenceInput, PredictPayload
#from cerulean_cloud.cloud_run_offset_tiles.handler import predict

In [ ]:
fastaiunet = {
    "type": "FASTAIUNET",
    "file_path": "",#"experiments/2024_03_06_18_14_31_7cls_rn101_pr256_z9_fastai_baseline_noamb/tracing_cpu_model.pt",
    "layers": ["VV"],
    "cls_map": {
        0: "BACKGROUND",
        1: "INFRA",
        2: "NATURAL",
        3: "COIN_VESSEL",
        4: "REC_VESSEL",
        5: "OLD_VESSEL",
        6: "BACKGROUND"  # HITL AMBIGUOUS, should never be output by inference_idx
    },  # inference_idx maps to class table
    "name": "ResNet101 Baseline Noamb",
    "tile_width_m": 40844,  # Used to calculate zoom
    "tile_width_px": 256,  # Used to calculate scale
    "epochs": 80,
    "thresholds": {
        "pixel_nms_thresh": 0.4,
        "bbox_score_thresh": 0.1,
        "poly_score_thresh": 0.01, # JONA Is this working correctly???
        "pixel_score_thresh": 0.35,
        "groundtruth_dice_thresh": 0.0
    },
    "backbone_size": 101,
    "zoom_level":9,
    "scale":2,
    # "pixel_f1": 0.0,  # TODO CALCULATE
    # "instance_f1": 0.0  # TODO CALCULATE
}

maskrcnn = dict(
    type="MASKRCNN",
    file_path="",#"experiments/2023_10_05_02_22_46_4cls_rnxt101_pr512_px1024_680min_maskrcnn_wd01/scripting_cpu_model.pt",
    layers=["VV", "ALL_255", "VESSEL"],
    cls_map={
        0: "BACKGROUND",
        1: "INFRA",
        2: "NATURAL",
        3: "VESSEL",
    },  # inference_idx maps to class table
    name="ResNext 101 hires56",
    tile_width_m=40844,
    tile_width_px=512,
    epochs=122,
    thresholds={
        "poly_nms_thresh": 0.2,
        'pixel_nms_thresh': 0.4,
        'bbox_score_thresh': 0.3,
        'poly_score_thresh': 0.1,
        'pixel_score_thresh': 0.5,
        'groundtruth_dice_thresh': 0.0
        },
    backbone_size=101,
    pixel_f1=0.461,
    instance_f1=0.47,
)

model_dict=maskrcnn

In [ ]:
class MockLayer:
    def __init__(self, short_name, source_url):
        self.short_name = short_name
        self.source_url = source_url
layers_fastaiunet = [MockLayer("VV","https://registry.opendata.aws/sentinel-1/")]
layers_maskrcnn = [MockLayer("VV","https://registry.opendata.aws/sentinel-1/"),
           MockLayer("INFRA","https://storage.googleapis.com/ceruleanml/aux_datasets/infra_locations_01_cogeo.tiff"),
           MockLayer("VESSEL", "https://gmtds.maplarge.com/public/ext/GMTDS/Main")]
layers = layers_maskrcnn

In [ ]:
async def get_titiler_client_and_offset_tiles(sentinel_scene, offset=.33):
    payload = OrchestratorInput(**sentinel_scene)
    TitilerClient_url = os.getenv('TITILER_URL')
    titiler_client = TitilerClient(url=TitilerClient_url)
    scene_bounds = await titiler_client.get_bounds(payload.sceneid)
    tiler = TMS
    base_tiles = list(tiler.tiles(*scene_bounds, [payload.zoom], truncate=False))
    offset_tile_bounds = offset_bounds_from_base_tiles(base_tiles, offset_amount=offset)
    return titiler_client, offset_tile_bounds

In [ ]:
scene_id = "S1A_IW_GRDH_1SDV_20230523T224049_20230523T224114_048667_05DA7A_91D1"
#scene_id = "S1A_IW_GRDH_1SDV_20231121T015923_20231121T015948_051309_0630CB_AADA"
test_scene = {"sceneid": scene_id , "zoom":9, "scale":2}
payload = OrchestratorInput(**test_scene)
titiler_client = TitilerClient(url=os.getenv('TITILER_URL'))
tiler = TMS
scene_bounds = await titiler_client.get_bounds(test_scene['sceneid'])
zoom = test_scene['zoom']
scale = test_scene['scale']
start_time = 'now-o-clock'

In [ ]:
"""Clients for other cloud run functions"""

import asyncio
import json
import os
import zipfile
from base64 import b64encode
from datetime import datetime
from io import BytesIO
from typing import List, Tuple

import httpx
import numpy as np
import rasterio
import skimage
from rasterio.enums import Resampling
from rasterio.io import MemoryFile
from rasterio.plot import reshape_as_raster
from rio_tiler.io import COGReader

In [ ]:
base_tiles = list(tiler.tiles(*scene_bounds, [zoom], truncate=False))

base_configs = {
    "model_dict": model_dict,
    "base_tiles": base_tiles,
    "layers": layers,
    "sceneid": payload.sceneid,
    "scale": payload.scale,
    "gamma_correction": 0.0,
    "tiling_offset": 0.0,
    "zoom": 9,
}

task_configs = [
    {"tiling_offset": 0.0},
    {"tiling_offset": 0.33},
    {"tiling_offset": 0.66},
]

task_list = [
    Task(**{**base_configs, **config}) for config in task_configs
]

In [ ]:
for task in task_list:
    await task.run_parallel_inference()
    #break
    task.postprocess_tileset()

In [ ]:
final_ensemble = Ensemblers.nms_ensemble(task_list)

In [ ]:
print(type(final_ensemble))

In [ ]:
final_ensemble

In [ ]:
properties_list = []
geometries = []
for red in final_ensemble['features']:
    # Extract geometry and properties
    geometry = shape(red["geometry"])
    properties = red["properties"]

    # Append to lists
    geometries.append(geometry)
    properties_list.append(properties)
# Create a GeoDataFrame
gdf = gpd.GeoDataFrame(properties_list, geometry=geometries)
gdf.plot()

In [ ]:
gdf.to_file('AADA_maskrcnn.geojson', driver='GeoJSON')